# CO2SITE — Task A: Monte Carlo capacity uncertainty and candidate q

> **How to derive scientifically justified indifference (q) and preference (p) thresholds for CO₂ storage capacity in PROMETHEE.**
>
> We aim to establish a reproducible procedure that can be applied to multiple CO₂ storage sites and assess whether it is sufficiently robust for implementation in the CO2SITE platform.

**Objective of this notebook:** compare simulated *storage-capacity distributions* from existing GEUS-derived examples, visualize the differences, calculate model-implied pairwise probabilities, and explore an uncertainty-derived **candidate q**. This notebook does **not** determine a validated PROMETHEE threshold, nor calculate p (Task B).

**Important:** This is a static volumetric capacity analysis. It does not establish dynamic injectivity or storage feasibility. Site comparisons are illustrative, and a capacity preference does not override geological safety constraints.

## 1 — Setup
Click **Runtime → Run all** in Google Colab. The notebook clones only the separate research branch; your `main` remains unchanged.

In [ ]:
# @title Setup and imports
import os, subprocess, sys
from pathlib import Path
REPO = Path("/content/ggg-co2-storage-eval")
if not REPO.exists():
    subprocess.check_call(["git", "clone", "--branch", "research/task-a-capacity-indifference",
                           "--single-branch", "https://github.com/AnaSoles/ggg-co2-storage-eval.git", str(REPO)])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])
# Source layout: the importable package is located in REPO/src, not REPO.
# Colab may not refresh editable-install paths until the kernel restarts.
source_dir = str(REPO / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from storageeval import load_site_csv, simulate
from storageeval.indifference import compare_all_sites, compare_capacity_samples, common_q_diagnostics
print("Ready. Loaded exploratory Task A analysis from:", REPO)

## 2 — Simulate several storage sites
Run **20,000 iterations** per site by default; change `ITERATIONS` to 100,000 for a more stable Monte Carlo estimate. Input distributions come from the repository's GEUS-based example CSV files. These are different published site/formation scenarios, not yet a validated common selection portfolio.

In [ ]:
# @title Simulate capacity
ITERATIONS = 20_000
BASE_SEED = 42
SITE_INPUTS = {
    "Rødby – Bunter": "data_rodby_inputs.csv",
    "Havnsø – Gassum": "data_havnso_s1_inputs.csv",
    "Inez – Haldager": "data_inez_haldager_inputs.csv",
}
results = {}
for k, (name, filename) in enumerate(SITE_INPUTS.items()):
    site = load_site_csv(REPO / "examples" / filename, name=name)
    results[name] = simulate(site, iterations=ITERATIONS, seed=BASE_SEED + k)
site_stats = pd.DataFrame([{
    "Site": name,
    "Mean (Mt)": float(r.capacity_mt.mean()),
    "SD (Mt)": float(r.capacity_mt.std(ddof=1)),
    "P90 (Mt)": float(np.quantile(r.capacity_mt, .10)),
    "P50 (Mt)": float(np.quantile(r.capacity_mt, .50)),
    "P10 (Mt)": float(np.quantile(r.capacity_mt, .90)),
} for name, r in results.items()])
display(site_stats.round(2))

## Figure 1 — Storage-capacity distributions
Each histogram shows uncertainty **within one site**. Wider distributions indicate more dispersed simulated estimates, but SD must be interpreted relative to capacity scale and model assumptions.

In [ ]:
# @title Plot capacity distributions
fig, ax = plt.subplots(figsize=(10, 5))
for name, r in results.items():
    ax.hist(r.capacity_mt, bins=70, density=True, alpha=0.34, label=name)
ax.set(xlabel="Simulated static storage capacity (Mt CO₂)",
       ylabel="Probability density", title="Figure 1 — Capacity uncertainty by site")
ax.legend(); ax.grid(alpha=.2); plt.show()

## Figure 2 — P90 / P50 / P10 capacity ranges
In storage-industry *exceedance notation*, P90 is the low (10th percentile) estimate and P10 is the high (90th percentile) estimate.

In [ ]:
# @title Plot capacity ranges
fig, ax = plt.subplots(figsize=(10, 4.5))
ys = np.arange(len(site_stats))
low = site_stats["P90 (Mt)"].to_numpy()
mid = site_stats["P50 (Mt)"].to_numpy()
high = site_stats["P10 (Mt)"].to_numpy()
ax.errorbar(mid, ys, xerr=[mid-low, high-mid], fmt="o", capsize=6, linewidth=2)
ax.set_yticks(ys, site_stats["Site"]); ax.invert_yaxis()
ax.set(xlabel="Storage capacity (Mt CO₂)", title="Figure 2 — P90–P10 intervals around P50")
ax.grid(axis="x", alpha=.25); plt.show()

## 3 — Compare two sites, iteration by iteration

For each pair, calculate the difference between the simulated capacity of Site B and Site A.

<details><summary>Show capacity-difference formula</summary>

$\\Delta_i = C_{B,i} - C_{A,i}$

</details>

The default function independently resamples each site's Monte Carlo distribution with replacement. This assumes the sites' uncertainties are independent. If they share uncertain regional assumptions, they need a **joint uncertainty model**; using the same seed is *not* sufficient to represent that dependence.

Example with five fictional draws:

| Draw | A (Mt) | B (Mt) | B − A |
|---|---:|---:|---:|
| 1 | 95 | 110 | +15 |
| 2 | 108 | 102 | −6 |
| 3 | 100 | 120 | +20 |
| 4 | 112 | 105 | −7 |
| 5 | 90 | 115 | +25 |

B exceeds A in 3 of 5 draws → estimated probability 60%. **This probability is not the q threshold.**

In [ ]:
# @title Compare two sites
NAMES = list(results)
SITE_A = NAMES[0]  # Change these two selections to inspect another pair
SITE_B = NAMES[1]
PAIR_SEED = 2026
a = results[SITE_A].capacity_mt
b = results[SITE_B].capacity_mt
rng = np.random.default_rng(PAIR_SEED)
delta = b[rng.integers(len(b), size=ITERATIONS)] - a[rng.integers(len(a), size=ITERATIONS)]
pair_summary = compare_capacity_samples(results[SITE_A], results[SITE_B],
                                         seed=PAIR_SEED, draws=ITERATIONS)
display(pd.DataFrame([pair_summary]).T.rename(columns={0: "Result"}))
print(f"P({SITE_B} > {SITE_A}) = {np.mean(delta > 0):.1%}")

## Figure 3 — Distribution of capacity differences
Values **above zero** favour Site B; values **below zero** favour Site A. This is the direct visual meaning of 'calculate the differences between simulated capacities'.

In [ ]:
# @title Plot pairwise differences
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(delta, bins=75, edgecolor="white", alpha=.85)
ax.axvline(0, color="black", linestyle="--", label="No difference")
lo, hi = np.quantile(delta, [.025, .975])
ax.axvline(lo, linestyle=":", label="2.5th percentile")
ax.axvline(hi, linestyle=":", label="97.5th percentile")
ax.set(xlabel=f"Δ capacity = {SITE_B} − {SITE_A} (Mt CO₂)",
       ylabel="Number of Monte Carlo comparisons",
       title="Figure 3 — Pairwise capacity difference distribution")
ax.legend(); ax.grid(alpha=.2); plt.show()

## 4 — Compare all site pairs
The **model-implied** comparison probabilities answer how often one sampled capacity exceeds another given the assumed input distributions. They are not confidence levels about actual site suitability.

In [ ]:
# @title Compare all sites
pairs = compare_all_sites(list(results.values()), confidence=.95, seed=2026)
pair_df = pd.DataFrame(pairs)
display(pair_df[["first_site", "second_site", "mean_difference_mt", "std_difference_mt",
                  "lower_difference_mt", "upper_difference_mt", "prob_second_greater",
                  "q_pair_candidate_mt"]].round(3))

## Figure 4 — Pairwise probability matrix
The cell in row A, column B estimates the probability that **B has greater simulated capacity than A**. The diagonal is deliberately blank.

In [ ]:
# @title Plot probability matrix
names = list(results)
matrix = np.full((len(names), len(names)), np.nan)
for r in pairs:
    i, j = names.index(r["first_site"]), names.index(r["second_site"])
    matrix[i,j] = r["prob_second_greater"]
    matrix[j,i] = r["prob_first_greater"]
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(np.ma.masked_invalid(matrix), vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(names)), names, rotation=30, ha="right")
ax.set_yticks(range(len(names)), names)
for i in range(len(names)):
    for j in range(len(names)):
        if i != j:
            ax.text(j,i,f"{matrix[i,j]:.0%}",ha="center",va="center",
                    color="white" if matrix[i,j]<.25 or matrix[i,j]>.75 else "black")
ax.set(xlabel="Compared site (B)", ylabel="Reference site (A)",
       title="Figure 4 — Model-implied P(capacity B > capacity A)")
fig.colorbar(im, ax=ax, label="Probability"); plt.tight_layout(); plt.show()

## 5 — From uncertainty to a candidate q (NOT a validated q)

This notebook proposes an **exploratory diagnostic**: for each pair, take half the width of the central 95% interval of the simulated difference distribution:

<details><summary>Show exploratory q formula</summary>

\[
q_{AB}^{\mathrm{exploratory}} = \frac{Q_{97.5\%}(\Delta_{AB})-Q_{2.5\%}(\Delta_{AB})}{2}.
\]

</details>

**Why exploratory?** This measures spread in possible *differences*; it does **not** establish a decision-maker's indifference threshold. Even if the difference interval excludes zero, its width can still be large. The choice of 95% is a convention used here to demonstrate uncertainty, not an established physical threshold.

Ordinary PROMETHEE typically expects a *single q per criterion* for all pairwise comparisons. Below we show how variable the pair-specific widths are and compute the 75th percentile as an **illustrative common-q diagnostic**, not a recommendation.

In [ ]:
# @title Explore candidate q
q_diag = common_q_diagnostics(pairs, quantile=.75)
display(pd.DataFrame([q_diag]).T.rename(columns={0: "Value"}))
labels = [f"{r['first_site']} vs {r['second_site']}" for r in pairs]
values = [r["q_pair_candidate_mt"] for r in pairs]
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(labels, values)
ax.axvline(q_diag["common_q_candidate_mt"], linestyle="--",
           label="Illustrative 75th-percentile common-q diagnostic")
ax.set(xlabel="Half-width of central 95% difference interval (Mt)",
       title="Figure 5 — Heterogeneity of exploratory pairwise uncertainty widths")
ax.legend(); ax.grid(axis="x", alpha=.2); plt.tight_layout(); plt.show()

## Figure 6 — How a candidate q would affect a PROMETHEE preference curve
Here **p is purely illustrative**, not calibrated (Task B). Changing q and p changes how much a difference counts. This final figure distinguishes a *statistical uncertainty diagnostic* from a *decision preference function*.

In [ ]:
# @title Plot PROMETHEE preference
q = float(q_diag["common_q_candidate_mt"])
p = max(q + 1, 2*q)  # illustrative only; NOT a scientifically derived p
x = np.linspace(0, max(1, 1.5*p), 500)
preference = np.where(x<=q, 0.0, np.where(x>=p, 1.0, (x-q)/(p-q)))
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(x, preference, linewidth=2)
ax.axvline(q, linestyle="--", label=f"Exploratory q = {q:.1f} Mt")
ax.axvline(p, linestyle=":", label=f"Illustrative p = {p:.1f} Mt (NOT calibrated)")
ax.set(xlabel="Capacity advantage of Site B over Site A (Mt CO₂)",
       ylabel="Single-criterion preference degree",
       ylim=(-.05,1.05), title="Figure 6 — PROMETHEE preference rule (illustrative)")
ax.legend(); ax.grid(alpha=.2); plt.show()

## Export / interpretation

- **What you can report now:** distributions per site, uncertainty of differences, model-implied probabilities, and dispersion of exploratory q candidates.
- **What you cannot yet report:** a scientifically calibrated universal q or p for CO2SITE.
- **Research next:** evaluate dependence between sites, comparability of capacity definitions, different confidence/quantile conventions, stakeholder/technical indifference judgements, and ranking sensitivity with alternative q values. Task B will address p.
- **Methodological warning:** the Monte Carlo distribution is a model-input uncertainty distribution, not a conventional sampling distribution for the mean. It is not valid to interpret its percentile interval automatically as a frequentist significance test.

To export the comparison table, run the following cell.

In [ ]:
# @title Export CSV
output = Path("/content/task_a_pairwise_comparisons.csv")
pair_df.to_csv(output, index=False)
print("Saved:", output)
# In Google Colab you can uncomment:
# from google.colab import files
# files.download(str(output))